# FINAL MODEL — Sinhala token-level offensive detection

Phase 2, Step 8. Full-train refit with the subword channel.

Trains **two** models and scores each on test once:

| model | channels | trainable | needs fastText? |
|---|---|---|---|
| **full** | word + subword | ~330k | yes |
| **minimal** | subword only | ~176k | **no** |

Everything is frozen from the Step 7b ablations. This notebook changes nothing;
it just runs the chosen configuration on all 7,500 training tweets.

---
## Colab setup — do this first

1. **Runtime → Change runtime type → Hardware accelerator: T4 GPU → Save**
2. Push your latest code to GitHub, then edit `REPO` in cell 3 below
3. Run every cell top to bottom

Expect **60–90 minutes** total on a T4. Colab wipes the session when it ends,
so download your results at the end (last cell). Keep the tab open or Colab
will disconnect you as idle.


## 1. Confirm the GPU


In [ ]:
!nvidia-smi -L
import torch
print('torch', torch.__version__, '| CUDA', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No GPU: Runtime -> Change runtime type -> T4 GPU'


## 2. Install packages

`datasets` is pinned below 3.0 because SOLD is a 2022-era dataset and newer
versions dropped support for its loading script.


In [ ]:
!pip install -q 'datasets<3.0.0' pytorch-crf sentencepiece 2>&1 | tail -1
from torchcrf import CRF
import sentencepiece
print('ready')


## 3. Get the code

**Edit REPO.** If the repo is private, skip this and use the upload cell below.


In [ ]:
REPO = 'https://github.com/YOUR-USERNAME/YOUR-REPO.git'   # <-- EDIT THIS

import os, shutil
if os.path.exists('project'): shutil.rmtree('project')
!git clone -q $REPO project
%cd project
!ls src/ notebooks/ tests/


*Private repo alternative: zip `src/`, `notebooks/` and `tests/`, then run this.*


In [ ]:
# from google.colab import files
# import zipfile, os
# up = files.upload()
# os.makedirs('project', exist_ok=True)
# zipfile.ZipFile(list(up)[0]).extractall('project')
# %cd project
# !ls


## 4. Safety checks — run BEFORE training

The alignment tests catch the one bug that would silently ruin everything:
labels are one per word, subword pieces are smaller than words, and if the
piece tensor ever has the wrong number of word-rows the labels shift against
the words. The model still trains, the loss still falls, and the score is
quietly wrong with nothing crashing.

**If either suite fails, stop. Do not train.**


In [ ]:
!python tests/test_metrics.py
!python tests/test_subword_alignment.py


## 5. Download the Sinhala fastText vectors

About 600 MB, two to three minutes. Needed for the **full** model; the
**minimal** model does not use them at all.

Do not unzip — the loader reads `.gz` directly.


In [ ]:
!mkdir -p embeddings results artifacts
![ -f embeddings/cc.si.300.vec.gz ] || wget -q --show-progress -O embeddings/cc.si.300.vec.gz https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.si.300.vec.gz
!ls -lh embeddings/


## 6. Sanity check — 1 seed, 3 epochs

Confirms the pipeline runs on GPU before committing to the full job.
The score will be poor because it barely trains. That is expected — you are
checking that it runs, not what it scores.


In [ ]:
!python notebooks/08_final_model.py --model full --seeds 1 --epochs 3 2>&1 | tail -20


## 7. THE FINAL RUN

Both models, 5 seeds each, 35 fixed epochs, test scored once.

The epoch budget of 35 is the median best-epoch from the bpe_1000 validation
runs (30, 30, 41, 35, 37). It came from validation, never from test — which is
what makes a fixed budget legitimate here.

**Do not re-run this with different settings if you dislike the number.** That
would be tuning on test. Whatever it gives is the result.


In [ ]:
!python notebooks/08_final_model.py --both 2>&1 | tee results/step8_final.txt


## 8. Optional: the last owed ablation

`--pooling mean` replaces the per-word BiLSTM with a simple average. It answers
whether piece **order** matters, and it may be considerably faster since the
subword encoder is now the bottleneck.

This is an ablation row, not a candidate for the headline model.


In [ ]:
!python notebooks/08_final_model.py --model full --pooling mean 2>&1 | tail -25


## 9. Download everything

Colab deletes the session when it closes. Save these and commit them.


In [ ]:
from google.colab import files
import os
for f in ['results/results_final.csv', 'results/step8_final.txt']:
    if os.path.exists(f):
        files.download(f)
    else:
        print('missing:', f)


---
## After it finishes

**Record in the README:** both models' F1 with standard deviations, precision
and recall, trainable parameter counts, and the change from the Phase 1
baseline of 0.5965.

**Check the precision/recall pattern.** The script prints a different message
depending on what it finds. If recall rose while precision held, the
unseen-morphology mechanism is confirmed on the full training split and you can
claim it. If precision fell instead, the model is simply firing more freely and
you must not claim that mechanism.

**Write it up honestly.** The headline is not "we nearly match XLM-R". It is:
*a lightweight model with a subword channel reaches this F1 at a small fraction
of the parameters, and the gain is precision-neutral and recall-driven.* The
mechanism is what makes a reviewer believe the number.

**Then Piece 2:** the joint sentence + token head.
